<div style="background-color:#EAEAEA;padding:20px;border-left:5px solid #6C757D;border-radius:6px;">
<table style="width:100%; border:none;"><tr style="border:none;">
<td style="border:none; vertical-align:top;">
<h1 style="font-size:32px; margin-top:0;">Master's Thesis</h1>
<hr style="margin:16px 0 22px 0;">
<p style="font-size:22px; line-height:1.5; margin:0;"><strong>Master's Degree in Advanced Physics</strong> - <strong>Universitat de Valencia</strong></p>
<p style="font-size:17px; margin-top:28px; margin-bottom:6px;">This notebook is part of the <strong>Master's Thesis (MSc Dissertation)</strong>:</p>
<div style="font-size:25px;font-weight:700;line-height:1.3;margin-top:14px;margin-bottom:26px;">Fast Simulation of Neutrino Oscillations in Matter</div>
<p style="font-size:14px; line-height:1.55;"><strong>Author</strong><br>Juan Ramon Diaz Santos - <a href="mailto:diazjuan@alumni.uv.es">diazjuan@alumni.uv.es</a></p>
<p style="font-size:14px; line-height:1.55;"><strong>Supervisors</strong><br>Roberto Ruiz de Austri Bazan - <a href="mailto:rruiz@ific.uv.es">rruiz@ific.uv.es</a><br>Michele Lucente - <a href="mailto:michele.lucente@unibo.it">michele.lucente@unibo.it</a></p>
<p style="font-size:14px; line-height:1.55; margin-bottom:0;"><strong>Date</strong><br>September 2026</p></td>
<td style="border:none;width:230px;padding-left:25px;text-align:right;vertical-align:top;"><img src="../logo_uv.png" alt="Universitat de Valencia" style="width:200px; margin-top:5px;"></td>
</tr></table></div>

# Benchmark 0: Summary
---
This notebook aggregates the performance results produced by `benchmark1_perturbative_vs_numerical.ipynb`, `benchmark2_tpeanuts_vs_peanuts.ipynb`, `benchmark3_tpeanuts_vs_nusquids.ipynb`, and `benchmark4_tpeanuts_profiler.ipynb`, and presents a compact dashboard of `tpeanuts`'s measured performance: perturbative-vs-numerical speedup, speedup against the legacy `peanuts` reference, speedup against `nuSQuIDS`, and where inside `tpeanuts` itself the time actually goes. Run notebooks 1-4 first to generate the required CSV files.

Unlike `nusquids0_summary.ipynb`/`validation_legacy0_summary.ipynb`/`intrinsic0_summary.ipynb`, this dashboard aggregates **timing/speedup** results, not precision comparisons -- there is no ppm/ppb pass/fail status here, only "faster" (speedup > 1) or "slower" (speedup < 1). Benchmarks 1-3 share a compatible per-comparison schema (one row per grid point, a `speedup` column, a case/section label) despite each using a different reference (numerical `tpeanuts`, legacy `peanuts`, `nuSQuIDS`), so Sections 3-5 normalize and aggregate them together. Benchmark 4 profiles `tpeanuts` against itself at the Python-function level (no external reference, no speedup), so it is aggregated separately in Section 6.

## Table of Contents

| # | Section |
|---|---|
| [0](#0.-Theoretical-Framework) | **Theoretical Framework** |
| [1](#1.-Libraries) | **Libraries** |
| [2](#2.-Paths-and-Configuration) | **Paths and Configuration** |
| [3](#3.-Load-Speedup-CSVs) | **Load Speedup CSVs** |
| [4](#4.-Global-Aggregation) | **Global Aggregation** |
| [5](#5.-Visualisation-and-Export) | **Visualisation and Export** |
| [6](#6.-Profiler-Bottlenecks-Benchmark-4) | **Profiler Bottlenecks (Benchmark 4)** |
| [7](#7.-Summary) | **Summary** |

## 0. Theoretical Framework
---
This section records the scope and provenance of the notebook.

**References**

No external references are cited in this notebook.

## 1. Libraries


In [ ]:
from __future__ import annotations

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tpeanuts.notebooks.notebookConfig import load_notebook_config
from tpeanuts.notebooks.notebooks_helper import save_and_show


## 2. Paths and Configuration

`load_notebook_config()` resolves the repository root and the shared output directory. `benchmark1`/`benchmark2`/`benchmark4` write to the flat `benchmark/` directory; `benchmark3` (the nuSQuIDS comparison, which needs the WSL `corsika8-venv` kernel) writes to its own `benchmark/nusquids/` subdirectory to avoid colliding with `benchmark2`'s per-section CSVs, which otherwise share the same generic filenames (`earth_probability_timing.csv`, `vacuum_flux_timing.csv`, ...). This notebook reads from both locations and writes its own exports to a `summary/` subdirectory inside the flat `benchmark/` root.

Each benchmark notebook uses a different schema and reference backend, so unlike `nusquids0_summary.ipynb`'s dynamic per-notebook prefix scan, the three speedup sources are mapped explicitly below (`SPEEDUP_SOURCES`) rather than inferred from filenames.

### 2.1 Paths

Repository-relative input and output locations are resolved here, ensuring reproducible execution without hidden state or external notebook dependencies.

### 2.2 Configuration

Physical parameters, numerical grids, precision, runtime context, and validation tolerances are fixed here for every subsequent calculation.

In [ ]:
config          = load_notebook_config()
BENCHMARK_ROOT  = config.output_dir("benchmark")
NUSQUIDS_ROOT   = config.output_dir("benchmark", "nusquids")
OUTPUT_DIR      = BENCHMARK_ROOT / "summary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SHOW_PLOTS      = config.show_plots

# Each entry: (notebook label, reference label, csv path, group column, speedup column,
# tpeanuts-time column, reference-time column). benchmark1 compares tpeanuts's own
# perturbative (analytical) method against its own numerical method -- not an external
# reference -- so "reference" there means "numerical", not a different library.
SPEEDUP_SOURCES = [
    {
        "notebook": "benchmark1_perturbative_vs_numerical",
        "reference": "numerical (tpeanuts)",
        "path": BENCHMARK_ROOT / "benchmark1_perturbative_vs_numerical.csv",
        "group_col": "case",
        "speedup_col": "speedup",
        "tpeanuts_col": "perturbative_s",
        "reference_col": "numerical_s",
    },
    {
        "notebook": "benchmark2_tpeanuts_vs_peanuts",
        "reference": "peanuts (legacy)",
        "path": BENCHMARK_ROOT / "all_performance_timing.csv",
        "group_col": "section",
        "speedup_col": "speedup_legacy_over_tpeanuts",
        "tpeanuts_col": "tpeanuts_best_s",
        "reference_col": "legacy_best_s",
    },
    {
        "notebook": "benchmark3_tpeanuts_vs_nusquids",
        "reference": "nuSQuIDS",
        "path": NUSQUIDS_ROOT / "all_performance_timing.csv",
        "group_col": "section",
        "speedup_col": "speedup_nusquids_over_tpeanuts",
        "tpeanuts_col": "tpeanuts_best_s",
        "reference_col": "nusquids_best_s",
    },
]

print(f"Benchmark root : {BENCHMARK_ROOT}")
print(f"nuSQuIDS root  : {NUSQUIDS_ROOT}")
print(f"Output dir     : {OUTPUT_DIR}")


## 3. Load Speedup CSVs

Loads each of the three speedup sources listed in Section 2 (skipping any that have not been generated yet), normalizes them to a common long-format schema (`notebook`, `reference`, `case`, `n_energy`, `n_second`, `tpeanuts_s`, `reference_s`, `speedup`), and concatenates them into one table.

When a source CSV is missing, this section reports it and continues with whatever is available -- matching the graceful-degradation pattern used by `nusquids0_summary.ipynb` when nuSQuIDS itself is unavailable.

In [ ]:
speedup_rows = []
missing = []
for source in SPEEDUP_SOURCES:
    if not source["path"].exists():
        missing.append(source["path"])
        continue
    df = pd.read_csv(source["path"])
    second_col = "n_angle" if "n_angle" in df.columns else "n_nadir"
    for _, row in df.iterrows():
        speedup_rows.append({
            "notebook": source["notebook"],
            "reference": source["reference"],
            "case": row[source["group_col"]],
            "n_energy": row.get("n_energy"),
            "n_second": row.get(second_col),
            "tpeanuts_s": row[source["tpeanuts_col"]],
            "reference_s": row[source["reference_col"]],
            "speedup": row[source["speedup_col"]],
        })

speedup_df = pd.DataFrame(speedup_rows)

if missing:
    print("Missing (not yet generated):")
    for path in missing:
        print(" ", path)
if speedup_df.empty:
    print("No speedup CSVs found. Run benchmark1-3 notebooks first.")
else:
    display(speedup_df.sort_values(["notebook", "case", "speedup"], ascending=[True, True, False]).head(30))
    print(f"... {len(speedup_df)} rows total across {speedup_df['notebook'].nunique()} notebook(s).")


## 4. Global Aggregation

Groups the per-grid-point speedup rows by notebook and case/section, reporting the median, minimum, and maximum speedup observed for each -- the same statistic `benchmark2`/`benchmark3`'s own internal Section 9/10 summaries compute, just combined across all three notebooks in one table. A second, coarser grouping by notebook alone gives the one-bar-per-notebook chart in Section 5.

**Expected results:**<br>
Speedup > 1 means `tpeanuts` is faster than the compared reference (numerical `tpeanuts`, legacy `peanuts`, or `nuSQuIDS`, depending on the notebook). Batched/vectorized workloads (Earth, solar-detector, atmosphere) are expected to show the largest speedups at large grid sizes; scalar per-point workloads (vacuum) are expected to show more modest ones.

In [ ]:
if not speedup_df.empty:
    case_summary = speedup_df.groupby(["notebook", "reference", "case"], as_index=False).agg(
        rows=("speedup", "count"),
        median_speedup=("speedup", "median"),
        min_speedup=("speedup", "min"),
        max_speedup=("speedup", "max"),
    )
    display(case_summary.sort_values("median_speedup", ascending=False))

    notebook_summary = speedup_df.groupby(["notebook", "reference"], as_index=False).agg(
        rows=("speedup", "count"),
        median_speedup=("speedup", "median"),
        min_speedup=("speedup", "min"),
        max_speedup=("speedup", "max"),
    )
    display(notebook_summary.sort_values("median_speedup", ascending=False))
else:
    case_summary = pd.DataFrame()
    notebook_summary = pd.DataFrame()
    print("No data available for aggregation.")


## 5. Visualisation and Export

Two charts: a per-notebook median-speedup bar chart with min/max error bars (matching the style already used inside `benchmark2`/`benchmark3`'s own Section 9/10 summaries, now comparing all three notebooks side by side), and a per-case/section breakdown so individual workflows (vacuum, Earth, solar-detector, atmosphere, ...) remain visible rather than averaged away.

Two aggregate CSV files are exported:

- `benchmark0_case_summary.csv` -- one row per (notebook, case/section) with median/min/max speedup.
- `benchmark0_notebook_summary.csv` -- one row per notebook with the same aggregate statistics.

In [ ]:
if not notebook_summary.empty:
    fig, ax = plt.subplots(figsize=(9.5, 4.5))
    ordered = notebook_summary.sort_values("median_speedup", ascending=True)
    xerr = np.array([
        ordered["median_speedup"] - ordered["min_speedup"],
        ordered["max_speedup"] - ordered["median_speedup"],
    ]).clip(min=0.0)
    colors = ["C2" if v > 1.0 else "C3" for v in ordered["median_speedup"]]
    ax.barh(ordered["notebook"], ordered["median_speedup"], xerr=xerr, color=colors,
            error_kw={"capsize": 4, "elinewidth": 1.2})
    ax.axvline(1.0, color="black", lw=1.2, alpha=0.6)
    ax.set_xscale("log")
    ax.set_xlabel("Median speedup vs reference (log scale)")
    ax.set_title("tpeanuts performance summary -- median speedup with min/max error bars")
    fig.tight_layout()
    save_and_show("benchmark0_fig1_notebook_speedup.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
    display(ordered[["notebook", "reference", "rows", "median_speedup", "min_speedup", "max_speedup"]])

    fig2, ax2 = plt.subplots(figsize=(11.0, max(4.0, 0.35 * len(case_summary))))
    case_ordered = case_summary.sort_values(["notebook", "median_speedup"], ascending=[True, True])
    labels = case_ordered["notebook"] + ": " + case_ordered["case"].astype(str)
    colors2 = ["C2" if v > 1.0 else "C3" for v in case_ordered["median_speedup"]]
    ax2.barh(labels, case_ordered["median_speedup"], color=colors2)
    ax2.axvline(1.0, color="black", lw=1.0, alpha=0.6)
    ax2.set_xscale("log")
    ax2.set_xlabel("Median speedup vs reference (log scale)")
    ax2.set_title("tpeanuts performance summary -- per case/section")
    fig2.tight_layout()
    save_and_show("benchmark0_fig2_case_speedup.png", fig2, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)

    case_summary.to_csv(OUTPUT_DIR / "benchmark0_case_summary.csv", index=False)
    notebook_summary.to_csv(OUTPUT_DIR / "benchmark0_notebook_summary.csv", index=False)
    print("Exported benchmark0_case_summary.csv and benchmark0_notebook_summary.csv")
else:
    print("No speedup CSV files found. Run benchmark1-3 notebooks first.")


## 6. Profiler Bottlenecks (Benchmark 4)

`benchmark4_tpeanuts_profiler.ipynb` profiles `tpeanuts` against itself (Python-function `cProfile`/`tottime`), not against an external reference, so it has no speedup column and is aggregated separately here. `profiler_tpeanuts_summary.csv` already ranks functions by how many profiled scenarios they appear in and their mean/max self-time share; this section just loads, displays, and charts it.

**Expected results:**<br>
Functions appearing in every scenario with a high mean share are the genuine, hard-to-avoid computational bottlenecks (e.g. eigenvalue solvers); functions appearing in only one or two scenarios are stage-specific costs.

In [ ]:
profiler_path = BENCHMARK_ROOT / "profiler_tpeanuts_summary.csv"
if profiler_path.exists():
    profiler_df = pd.read_csv(profiler_path)
    display(profiler_df.head(20))

    top = profiler_df.sort_values("mean_share", ascending=True).tail(15)
    fig, ax = plt.subplots(figsize=(9.5, 0.38 * len(top) + 1.5))
    ax.barh(top["function"], top["mean_share"], color="C4")
    ax.set_xlabel("Mean self-time share across profiled scenarios")
    ax.set_title("Ranked tpeanuts bottlenecks (benchmark4_tpeanuts_profiler)")
    fig.tight_layout()
    save_and_show("benchmark0_fig3_profiler_bottlenecks.png", fig, output_dir=OUTPUT_DIR, show_plots=SHOW_PLOTS)
else:
    print("Missing (not yet generated):", profiler_path)
    print("Run benchmark4_tpeanuts_profiler.ipynb first.")


## 7. Summary
---
This notebook documented **Benchmark 0: Summary**, including its methodology, principal calculations and resulting diagnostics. The preceding sections contain the detailed numerical outputs and visual checks needed to interpret the result.
